# O ranking está aritmeticamente correto e clinicamente absurdo

PRR sobre uma 2×2 de relatórios distintos, com a lista de exclusão
aplicada e um piso de três co-relatos. O critério de aceite desta
tarefa era ler o topo da tabela a olho.

In [1]:
import sys

sys.path.insert(0, "../src")

import duckdb
import pandas

from hindsight.analysis.prr import top_pairs

PARQUET = "../data/parquet"
PARTITION = f"{PARQUET}/year=2025/quarter=1/part=0001-of-0028"
EXCLUSIONS = "../reference/excluded_terms.csv"

ranked = lambda **kwargs: top_pairs(root=PARQUET, exclusions=EXCLUSIONS, **kwargs)

In [2]:
pairs = pandas.DataFrame(ranked(limit=12))

pairs[["drug", "event", "a", "b", "c", "d", "prr", "chi2"]]

,drug,event,a,b,c,d,prr,chi2
0,DESOGESTREL\ETHINYL ESTRADIOL,X-ray abnormal,4,1,1,11994,9596.000000,5877.899312
1,DESOGESTREL\ETHINYL ESTRADIOL,General symptom,3,2,1,11994,7197.000000,3747.812005
2,DESOGESTREL\ETHINYL ESTRADIOL,Pustular psoriasis,3,2,1,11994,7197.000000,3747.812005
3,MYOCHRYSINE,Onychomadesis,8,6,1,11985,6849.142857,5352.407455
4,BUTRANS,Onychomycosis,9,9,1,11981,5991.000000,4810.901089
5,GOLD SODIUM THIOMALATE,Onychomycosis,9,9,1,11981,5991.000000,4810.901089
6,BERINERT,Onychomycosis,9,11,1,11979,5391.000000,4328.832736
7,BUTRANS,Onychomadesis,8,10,1,11981,5325.333333,4161.037821
8,NADOLOL,Proctitis,3,4,1,11992,5139.857143,2676.026266
9,NADOLOL,Vaginal flatulence,3,4,1,11992,5139.857143,2676.026266


Micose de unha num adesivo de buprenorfina. Um sal de ouro injetável.
Um inibidor de C1-esterase. Nada disso é farmacologia.

A causa usual de uma primeira linha implausível são marginais erradas,
então a célula seguinte recalcula uma 2×2 à mão.

In [3]:
duckdb.sql(f'''
    WITH exposure AS (
        SELECT DISTINCT safetyreportid
        FROM '{PARTITION}/report_drug.parquet'
        WHERE medicinalproduct = 'BUTRANS'
    ),
    occurrence AS (
        SELECT DISTINCT safetyreportid
        FROM '{PARTITION}/report_reaction.parquet'
        WHERE reactionmeddrapt = 'Onychomycosis'
    )
    SELECT
        (SELECT count(*) FROM exposure  SEMI JOIN occurrence USING (safetyreportid)) AS a,
        (SELECT count(*) FROM exposure ANTI JOIN occurrence USING (safetyreportid)) AS b,
        (SELECT count(*) FROM occurrence ANTI JOIN exposure USING (safetyreportid)) AS c
''')

┌───────┬───────┬───────┐
│   a   │   b   │   c   │
│ int64 │ int64 │ int64 │
├───────┼───────┼───────┤
│     9 │     9 │     1 │
└───────┴───────┴───────┘

## a = 9, b = 9, c = 1, d = 11.981 — e somam exatamente 12.000

A consulta está certa. A resposta continua absurda, que é a versão
mais difícil dessa falha: nada na aritmética vai te dizer o que está
errado.

Evans (PRR ≥ 2, χ² ≥ 4, a ≥ 3) também não ajuda. Mantém 85% da
tabela, e o χ² é *grande* justamente nesses pares porque a contagem
esperada é 0,015. As duas estatísticas concordam com entusiasmo
sobre a mesma entrada ruim.

In [4]:
every_pair = ranked(limit=None)
flagged = [pair for pair in every_pair if pair.signal]

len(every_pair), len(flagged), len(flagged) / len(every_pair)

(28540, 24299, 0.8514015416958655)

Um limiar não separa uma duplicata de um sinal. As duas estatísticas
são funções de `a`, e `a` é o que está errado. O notebook 03 vai ler
os relatórios.